# Fault Injection & Detection Evaluation

Steps 3-4: inject gradual drift and stuck-at faults into the held-out test engines (never used for training or calibration), run them through the consistency models + thresholds from `sensor_consistency.ipynb`, and measure detection rate, false-alarm rate, detection delay, and sensor identification accuracy.

Run `sensor_consistency.ipynb` first if `consistency_models.pkl` doesn't exist yet.

In [1]:
import pandas as pd
import numpy as np
import joblib

artifact = joblib.load("consistency_models.pkl")
models = artifact["models"]
sigmas = artifact["sigmas"]
thresholds = artifact["thresholds"]
MONITORED_SENSORS = artifact["monitored_sensors"]
FEATURES = artifact["features"]
PERSISTENCE_WINDOW = artifact["persistence_window"]
PERSISTENCE_MIN_COUNT = artifact["persistence_min_count"]

print("Monitored sensors:", MONITORED_SENSORS)

Monitored sensors: ['Sensor_9', 'Sensor_14', 'Sensor_11', 'Sensor_12', 'Sensor_8', 'Sensor_13', 'Sensor_4', 'Sensor_7', 'Sensor_15', 'Sensor_20', 'Sensor_21', 'Sensor_17', 'Sensor_2', 'Sensor_3']


In [2]:
df = pd.read_csv("Dataset/processed_train.csv")

# Same seed, same shuffle, same slicing as sensor_consistency.ipynb -- this
# reproduces the identical train/val/test split deterministically, so
# test_df here is guaranteed to be the same 15 engines that notebook never
# trained on or calibrated against.
np.random.seed(42)
engine_ids = df["Engine_ID"].unique()
np.random.shuffle(engine_ids)
test_ids = engine_ids[85:]

test_df = df[df["Engine_ID"].isin(test_ids)].copy()
print("Evaluation engines:", test_df["Engine_ID"].nunique())

Evaluation engines: 15


## Fault injection functions

In [3]:
def inject_gradual_drift(engine_data, sensor, start_cycle, severity_per_cycle):
    """Adds a linear ramp to `sensor` from `start_cycle` onward. Positive
    severity drifts up, negative drifts down. Only this one sensor column
    changes -- RUL and every other column stay untouched, because a silent
    fault is a wrong *reading*, not a real change in engine health."""
    engine_data = engine_data.copy()
    mask = engine_data["Cycle"] >= start_cycle
    steps = (engine_data.loc[mask, "Cycle"] - start_cycle + 1).values
    engine_data.loc[mask, sensor] = engine_data.loc[mask, sensor].values + severity_per_cycle * steps
    return engine_data


def inject_stuck_at(engine_data, sensor, start_cycle):
    """Freezes `sensor` at its last clean value from `start_cycle` onward."""
    engine_data = engine_data.copy()
    mask = engine_data["Cycle"] >= start_cycle
    frozen_value = engine_data.loc[engine_data["Cycle"] == start_cycle - 1, sensor]
    if len(frozen_value) == 0:
        frozen_value = engine_data.loc[engine_data["Cycle"] == start_cycle, sensor]
    frozen_value = frozen_value.values[0]
    engine_data.loc[mask, sensor] = frozen_value
    return engine_data

## Persistence rule (same definition as `sensor_consistency.ipynb`)

In [4]:
def check_persistence(flags, window=7, min_count=5):
    flags = np.asarray(flags).astype(int)
    rolling_sum = pd.Series(flags).rolling(window, min_periods=1).sum().values
    return rolling_sum >= min_count

## Detection function

Runs every monitored sensor's consistency model over one engine's cycles. When more than one sensor is over threshold and persistent at the same cycle, the one with the largest normalized residual is reported as the suspected sensor.

In [5]:
def run_detection(engine_data, models, sigmas, thresholds, monitored_sensors, features,
                   persistence_window=7, persistence_min_count=5, search_from_cycle=None):
    engine_data = engine_data.sort_values("Cycle").reset_index(drop=True)

    normalized_residuals = pd.DataFrame(index=engine_data.index)
    for sensor in monitored_sensors:
        predictors = [f for f in features if f != sensor]
        pred = models[sensor].predict(engine_data[predictors])
        residual = np.abs(engine_data[sensor].values - pred) / sigmas[sensor]
        normalized_residuals[sensor] = residual

    flags = normalized_residuals.apply(lambda col: col > thresholds[col.name], axis=0)
    persistent = flags.apply(lambda col: check_persistence(col, persistence_window, persistence_min_count), axis=0)

    any_persistent = persistent.any(axis=1)

    # Only count a detection at or after the fault's true onset -- a
    # persistence hit earlier than this can only be a false alarm on the
    # still-clean portion of the run, and must never be reported as
    # detecting a fault that hasn't started yet. Left as None for clean
    # (false-alarm-check) runs, which should still search the whole engine.
    if search_from_cycle is not None:
        any_persistent = any_persistent & (engine_data["Cycle"] >= search_from_cycle)

    detected_cycle = None
    identified_sensor = None
    if any_persistent.any():
        first_idx = any_persistent.idxmax()  # first True (first persistent detection)
        detected_cycle = int(engine_data.loc[first_idx, "Cycle"])

        persistent_sensors_at_idx = persistent.columns[persistent.loc[first_idx].values].tolist()
        identified_sensor = normalized_residuals.loc[first_idx, persistent_sensors_at_idx].idxmax()

    return detected_cycle, identified_sensor

## Evaluation sweep

For every test engine: one clean run (false-alarm check), plus every monitored sensor × fault type × severity × onset combination. Severity doesn't apply to stuck-at faults, so both severity passes for `stuck_at` are identical runs -- harmless, just a little redundant, kept for a uniform loop structure.

In [6]:
FAULT_TYPES = ["drift_up", "drift_down", "stuck_at"]
SEVERITIES = {"mild": 0.002, "severe": 0.01}   # per-cycle drift added, in the sensor's own normalized units
ONSET_CYCLES = [50, 100]

results = []

for engine_id, engine_data in test_df.groupby("Engine_ID"):
    engine_data = engine_data.sort_values("Cycle").reset_index(drop=True)
    if engine_data["Cycle"].max() <= max(ONSET_CYCLES) + PERSISTENCE_WINDOW:
        continue  # engine too short to test a late onset meaningfully

    # --- clean run: false-alarm check ---
    detected_cycle, identified_sensor = run_detection(
        engine_data, models, sigmas, thresholds, MONITORED_SENSORS, FEATURES,
        PERSISTENCE_WINDOW, PERSISTENCE_MIN_COUNT
    )
    results.append({
        "Engine_ID": engine_id, "Fault_type": "none", "Sensor": None,
        "Severity": None, "Onset": None,
        "False_alarm": detected_cycle is not None,
        "Detected": None, "Detection_delay": None, "Correct_sensor": None
    })

    # --- fault runs ---
    for sensor in MONITORED_SENSORS:
        for fault_type in FAULT_TYPES:
            for severity_name, severity_value in SEVERITIES.items():
                for onset in ONSET_CYCLES:

                    if fault_type == "drift_up":
                        faulted = inject_gradual_drift(engine_data, sensor, onset, severity_value)
                    elif fault_type == "drift_down":
                        faulted = inject_gradual_drift(engine_data, sensor, onset, -severity_value)
                    else:
                        faulted = inject_stuck_at(engine_data, sensor, onset)

                    detected_cycle, identified_sensor = run_detection(
                        faulted, models, sigmas, thresholds, MONITORED_SENSORS, FEATURES,
                        PERSISTENCE_WINDOW, PERSISTENCE_MIN_COUNT, search_from_cycle=onset
                    )

                    detected = detected_cycle is not None
                    delay = (detected_cycle - onset) if detected else None
                    correct = (identified_sensor == sensor) if detected else None

                    results.append({
                        "Engine_ID": engine_id, "Fault_type": fault_type, "Sensor": sensor,
                        "Severity": severity_name, "Onset": onset,
                        "False_alarm": None,
                        "Detected": detected, "Detection_delay": delay, "Correct_sensor": correct
                    })

results_df = pd.DataFrame(results)
print("Total evaluation runs:", len(results_df))

Total evaluation runs: 2535


## Headline metrics

In [7]:
clean_runs = results_df[results_df["Fault_type"] == "none"]
fault_runs = results_df[results_df["Fault_type"] != "none"]

false_alarm_rate = clean_runs["False_alarm"].mean()
detection_rate = fault_runs["Detected"].mean()
mean_detection_delay = fault_runs.loc[fault_runs["Detected"] == True, "Detection_delay"].mean()
sensor_id_accuracy = fault_runs.loc[fault_runs["Detected"] == True, "Correct_sensor"].mean()

print(f"False-alarm rate (clean engines):     {false_alarm_rate:.1%}")
print(f"Detection rate (all injected faults): {detection_rate:.1%}")
print(f"Mean detection delay (cycles):        {mean_detection_delay:.1f}")
print(f"Sensor identification accuracy:       {sensor_id_accuracy:.1%}")

False-alarm rate (clean engines):     0.0%
Detection rate (all injected faults): 89.4%
Mean detection delay (cycles):        65.7
Sensor identification accuracy:       95.6%


## Breakdown by fault type and severity

In [8]:
breakdown = fault_runs.groupby(["Fault_type", "Severity"]).agg(
    Detection_rate=("Detected", "mean"),
    Mean_delay=("Detection_delay", "mean"),
    Sensor_ID_accuracy=("Correct_sensor", "mean")
).reset_index()

print(breakdown)

   Fault_type Severity Detection_rate  Mean_delay Sensor_ID_accuracy
0  drift_down     mild       0.795238   80.185629           0.973054
1  drift_down   severe            1.0   21.583333           0.992857
2    drift_up     mild       0.795238   79.667665           0.952096
3    drift_up   severe            1.0   21.207143            0.97381
4    stuck_at     mild       0.885714  102.879032           0.919355
5    stuck_at   severe       0.885714  102.879032           0.919355


In [9]:
results_df.to_csv("fault_detection_results.csv", index=False)
print("Saved fault_detection_results.csv")

Saved fault_detection_results.csv
